# Задача
- Оценить результаты А/Б-теста, в ходе которого проверялась новая ранжирующая формула в рекомендациях добавления в корзину на маркетплейсе.

# Описание имеющихся данных
В эксерименте разбиение по группам происходит на уровне пользователей (плановое разбиение - 50/50). Ниже - описание имеющихся полей в таблице
| Поле | Описание |
| --- | --- |
| user_id | Идентификатор пользователя |
| experiment_group	| Назначенная группа (control/treatment)|
| assignment_date | Дата назначения группы |
| was_exposed | Увидел ли пользователь блок рекомендаций |
| sessions | Количество сессий пользователя |
| orders | Количество заказов пользователя |
| gmv | Стоимость заказанных товаров |
| gross_margin | Выручка с пользователя за вычетом себестоимости товаров |
| promo_cost | Сколько маркетплейс потратил на маркетинг для этого пользователя |
| delivery_cost | Сколько маркетплейс потратил на доставку для этого пользователя |
| cancellations | Количество отмен заказов |
| late_deliveries | Количество заказов, для которых доставка задержалась |
| support_contacts | Количество обращений в поддержку |
| recommendation_impressions | Количество показанных товаров в блоке рекомендаций |
| recommendation_clicks | Количество кликов на товары из блока рекомендаций|
| recommended_add_to_cart | Количество добавлений в корзину из блока рекомендаций |
| recommended_purchases | Количество покупок после добавления из блока рекомендаций |
| preperiod_orders, preperiod_gmv | Показатели пользователя до начала эксперимента |
| platform | Платформа устройства пользователя (ios / android) |
| region | Регион (Москва / СПб / другое) |

# Цель исследования
- Проверить корректность данных

- Зафиксировать целевую, вспомогательные и контрольные метрики для А/Б-теста

- Решить, каких пользователей рассматривать в ходе эксперимента (применять ли маску `was_expanded` = 1)

- Предложить экономическую интерпретацию результатов эксперимента

- По имеющимся данным сделать вывод о результате эксперимента и принять решение по введению новой ранжирующей формулы

## Импорты, константы и загрузка данных

In [ ]:
import pandas as pd
import numpy as np
import scipy
from scipy.stats import chisquare, ttest_ind, chi2_contingency, norm, t

In [ ]:
for lib in [pd, np, scipy]:
    print(f"{lib.__name__}: {lib.__version__}")

pandas: 2.2.3
numpy: 2.1.3
scipy: 1.16.3


In [ ]:
data = pd.read_csv('recomendation_ab_test_data.csv', parse_dates=['assignment_date'])

In [ ]:
data.head()

,user_id,experiment_group,assignment_date,was_exposed,sessions,orders,gmv,gross_margin,promo_cost,delivery_cost,...,late_deliveries,support_contacts,recommendation_impressions,recommendation_clicks,recommended_add_to_cart,recommended_purchases,preperiod_orders,preperiod_gmv,platform,region
0,1019963,treatment,2026-10-20,0,2,1,1661.61,410.55,97.21,360.12,...,0,0,0,0,0,0,1,1702.10,ios,other
1,1008845,control,2026-10-24,0,6,1,2347.96,664.86,83.49,264.20,...,0,0,0,0,0,0,1,1140.58,ios,spb
2,1011207,treatment,2026-10-22,1,11,1,2094.89,538.65,96.33,354.22,...,0,0,18,0,0,0,1,1328.43,android,other
3,1024701,treatment,2026-10-17,1,8,0,0.00,0.00,0.00,0.00,...,0,0,11,1,0,0,0,0.00,android,other
4,1008974,treatment,2026-10-11,1,6,1,1166.62,296.60,127.31,326.68,...,0,0,8,0,0,0,1,1575.82,android,other


In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30008 entries, 0 to 30007
Data columns (total 21 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   user_id                     30008 non-null  int64         
 1   experiment_group            30008 non-null  object        
 2   assignment_date             30008 non-null  datetime64[ns]
 3   was_exposed                 30008 non-null  int64         
 4   sessions                    30008 non-null  int64         
 5   orders                      30008 non-null  int64         
 6   gmv                         30008 non-null  float64       
 7   gross_margin                30008 non-null  float64       
 8   promo_cost                  30008 non-null  float64       
 9   delivery_cost               30008 non-null  float64       
 10  cancellations               30008 non-null  int64         
 11  late_deliveries             30008 non-null  int64     

## Проверка данных

In [ ]:
# Посмотрим на дубликаты
counts = data['user_id'].value_counts()
not_unique_users = counts[counts > 1].index

In [ ]:
duplicates = data[data['user_id'].isin(list(not_unique_users))].sort_values(by='user_id')
duplicates

,user_id,experiment_group,assignment_date,was_exposed,sessions,orders,gmv,gross_margin,promo_cost,delivery_cost,...,late_deliveries,support_contacts,recommendation_impressions,recommendation_clicks,recommended_add_to_cart,recommended_purchases,preperiod_orders,preperiod_gmv,platform,region
8040,1000237,treatment,2026-10-25,0,10,2,3826.83,1046.97,214.49,689.98,...,0,0,0,0,0,0,1,1511.87,android,other
14758,1000237,treatment,2026-10-25,0,10,2,3826.83,1046.97,214.49,689.98,...,0,0,0,0,0,0,1,1511.87,android,other
16707,1001316,control,2026-10-06,0,3,0,0.00,0.00,0.00,0.00,...,0,0,0,0,0,0,1,1456.94,ios,other
21285,1001316,control,2026-10-06,0,3,0,0.00,0.00,0.00,0.00,...,0,0,0,0,0,0,1,1456.94,ios,other
20966,1003711,treatment,2026-10-12,1,6,1,2323.42,616.74,97.22,283.10,...,0,0,9,1,1,0,2,2658.40,android,spb
7949,1003711,treatment,2026-10-12,1,6,1,2323.42,616.74,97.22,283.10,...,0,0,9,1,1,0,2,2658.40,android,spb
216,1004850,treatment,2026-10-07,1,7,1,1395.19,374.42,96.69,265.78,...,0,0,15,2,1,1,3,5733.81,ios,moscow
5604,1004850,treatment,2026-10-07,1,7,1,1395.19,374.42,96.69,265.78,...,0,0,15,2,1,1,3,5733.81,ios,moscow
15880,1010695,control,2026-10-25,1,7,0,0.00,0.00,0.00,0.00,...,0,0,18,1,0,0,2,2876.45,ios,spb
26772,1010695,control,2026-10-25,1,7,0,0.00,0.00,0.00,0.00,...,0,0,18,1,0,0,2,2876.45,ios,spb


In [ ]:
# Видим, что тут просто у 8 пользователей дублируются записи, удалим дубликаты
data = data.drop_duplicates()

In [ ]:
# Посмотрим, где есть пропуски
data.isna().sum()

,0
user_id,0
experiment_group,0
assignment_date,0
was_exposed,0
sessions,0
orders,0
gmv,0
gross_margin,0
promo_cost,0
delivery_cost,0


In [ ]:
# Посмотрим, равномерно ли распределены пропуски по группам
data[data['platform'].isna()]['experiment_group'].value_counts()

,count
experiment_group,
treatment,14
control,10


In [ ]:
# Пропуски только в колонке platform и их немного, при этом по контролю/тесту
# они разбиты достаточно равномерно. Просто заполним их отдельной категорией
data['platform'] = data['platform'].fillna('unknown')

In [ ]:
# Проверим логику воронки
logic_errors = {
    'impressions without exposed': (data['recommendation_impressions'] > 0) > data['was_exposed'],
    'clicks > impressions': data['recommendation_clicks'] > data['recommendation_impressions'],
    'add_to_cart > clicks': data['recommended_add_to_cart'] > data['recommendation_clicks'],
    'purchases > add_to_cart': data['recommended_purchases'] > data['recommended_add_to_cart'],
    'cancellations > orders': data['cancellations'] > data['orders'],
    'late_deliveries > orders': data['late_deliveries'] > data['orders'],
}

In [ ]:
for error, series_error in logic_errors.items():
    print(f"{series_error.sum()} errors of type {error}")

0 errors of type impressions without exposed
4 errors of type clicks > impressions
0 errors of type add_to_cart > clicks
0 errors of type purchases > add_to_cart
0 errors of type cancellations > orders
0 errors of type late_deliveries > orders


In [ ]:
data[logic_errors['clicks > impressions']]

,user_id,experiment_group,assignment_date,was_exposed,sessions,orders,gmv,gross_margin,promo_cost,delivery_cost,...,late_deliveries,support_contacts,recommendation_impressions,recommendation_clicks,recommended_add_to_cart,recommended_purchases,preperiod_orders,preperiod_gmv,platform,region
6661,1006165,treatment,2026-10-14,1,10,0,0.00,0.00,0.00,0.00,...,0,0,18,19,1,1,1,1808.83,ios,spb
13954,1006905,control,2026-10-17,1,19,6,12778.27,3097.67,629.89,1805.90,...,0,0,29,30,1,1,9,12846.80,ios,other
15109,1021054,control,2026-10-13,1,9,1,1016.59,259.22,82.01,319.09,...,0,0,12,13,0,0,2,3200.42,ios,other
29219,1027020,treatment,2026-10-10,1,5,2,4067.05,968.06,241.15,640.09,...,0,0,9,10,0,0,0,0.00,ios,other


Логика воронки нарушается в 4 записях на этапе "показы $\to$ клики". Похоже на какую-то техническую ошибку в сборе данных, ведь везде кликов ровно на 1 больше. На всякий случай исключим данные из рассмотрения, от четырех юзеров сильно не убудет

In [ ]:
data = data[~logic_errors['clicks > impressions']]

## Проверка эксперимента

In [ ]:
# Проверим пропроции разбиения
group_counts = data['experiment_group'].value_counts()
group_counts

,count
experiment_group,
control,15007
treatment,14989


In [ ]:
expected = (group_counts.sum() / 2, group_counts.sum() / 2)
chi2, p_srm = chisquare(group_counts, expected)
print(f"P-value of SRM = {p_srm}")

P-value of SRM = 0.9172249174353291


P-value большой, значит с разбиением всё хорошо.

Проверим теперь баланс предпериодных признаков

In [ ]:
# Числовые фичи
pd.concat(
    [
        data[data['experiment_group'] == 'control'][['preperiod_orders', 'preperiod_gmv']].describe().rename(
            columns={
                'preperiod_orders': 'preperiod_orders_control',
                'preperiod_gmv': 'preperiod_gmv_control',
            }
        ),
        data[data['experiment_group'] == 'treatment'][['preperiod_orders', 'preperiod_gmv']].describe().rename(
            columns={
                'preperiod_orders': 'preperiod_orders_treatment',
                'preperiod_gmv': 'preperiod_gmv_treatment',
            }
    )], axis=1
).loc[
    ['mean', 'std', '25%', '50%', '75%']
].reindex(
    columns=['preperiod_orders_control', 'preperiod_orders_treatment', 'preperiod_gmv_control', 'preperiod_gmv_treatment']
)

,preperiod_orders_control,preperiod_orders_treatment,preperiod_gmv_control,preperiod_gmv_treatment
mean,1.205837,1.195010,2029.734286,2023.645249
std,1.368501,1.378351,2457.862209,2506.469201
25%,0.000000,0.000000,0.000000,0.000000
50%,1.000000,1.000000,1515.190000,1493.800000
75%,2.000000,2.000000,2994.300000,2941.590000


In [ ]:
# Категориальные фичи
pd.concat(
    [
        data[data['experiment_group'] == 'control'][['platform', 'region']].value_counts(sort=False, normalize=True).rename(
            'propotion_control'
        ),
        data[data['experiment_group'] == 'treatment'][['platform', 'region']].value_counts(sort=False, normalize=True).rename(
            'propotion_treatment'
        ),
    ], axis=1
)

propotion_control  propotion_treatment
platform region                                        
android  moscow           0.217032             0.216959
         other            0.186846             0.197478
         spb              0.143266             0.136433
ios      moscow           0.175252             0.174862
         other            0.162924             0.158850
         spb              0.114013             0.114484
unknown  moscow           0.000200             0.000600
         other            0.000400             0.000200
         spb              0.000067             0.000133

In [ ]:
def calculate_smd(group1, group2):
    mean1, mean2 = group1.mean(), group2.mean()
    n1, n2 = len(group1), len(group2)
    std = np.sqrt((np.var(group1) * n1 + np.var(group2) * n2) / (n1 + n2))
    return abs(mean1 - mean2) / std

Для числовых признаков проверим SMD между control и treatment

In [ ]:
smd_ord = calculate_smd(
    data[data['experiment_group'] == 'control']['preperiod_orders'],
    data[data['experiment_group'] == 'treatment']['preperiod_orders']
)
print(f"Standardized mean difference for preperiod orders = {smd_ord:.3f}")
if smd_ord < 0.1:
    print("Balance is OK")
else:
  print("Some problems with balance in preperiod feature")

Standardized mean difference for preperiod orders = 0.008
Balance is OK


In [ ]:
smd_gmv = calculate_smd(
    data[data['experiment_group'] == 'control']['preperiod_gmv'],
    data[data['experiment_group'] == 'treatment']['preperiod_gmv']
)
print(f"Standardized mean difference for preperiod GMV = {smd_gmv:.3f}")
if smd_gmv < 0.1:
    print("Balance is OK")
else:
  print("Some problems with balance in preperiod feature")

Standardized mean difference for preperiod GMV = 0.002
Balance is OK


Для категориальных признаков применим $\chi^2$-тест для проверки независимости группы от признака

In [ ]:
_, platform_p_val, _, _ = chi2_contingency(pd.crosstab(data['platform'], data['experiment_group']))
print(f"P-value for platform chi_square test = {platform_p_val:.4f}")
if platform_p_val > 0.05:
    print("Balance is OK")
else:
    print("Some problems with balance in preperiod feature")

P-value for platform chi_square test = 0.5693
Balance is OK


In [ ]:
_, region_p_val, _, _ = chi2_contingency(pd.crosstab(data['region'], data['experiment_group']))
print(f"P-value for region chi_square test = {region_p_val:.4f}")
if region_p_val > 0.05:
    print("Balance is OK")
else:
    print("Some problems with balance in preperiod feature")

P-value for region chi_square test = 0.3629
Balance is OK


Доли по всем предпериодным признакам сбалансированы: подтвердили это и визуально, и статистически

## Метрики

Целевая метрика будет:
| Метрика | Аудитория | Знаменатель | Единица анализа |
| --- | --- | --- | --- |
| $\frac{\text{GMV}}{\text{users}}$ | Все пользователи | Пользователи | Пользователи|

Хотим, чтобы наше изменение увеличивало выручку в пересчёте на пользователя

Контрольные метрики:

| Метрика | Аудитория | Знаменатель | Единица анализа |
| --- | --- | --- | --- |
|$\frac{(\text{gross_margin} - \text{promo_cost} - \text{delivery_cost})}{users}$ | Все пользователи | Пользователи | Пользователи|
|$\frac{\text{cancellations}}{\text{orders}}$ | Пользователи с хотя бы 1 заказом | Заказы | Пользователи |
|$\frac{\text{late_deliveries}}{\text{orders}}$ | Пользователи с хотя бы 1 заказом | Заказы | Пользователи |
|$\frac{\text{support_contacts}}{\text{orders}}$ | Пользователи с хотя бы 1 заказом | Заказы | Пользователи |

Первая метрика - прибыль с пользователя за вычетом расходов на маркетинг и доставку, нам важно, чтобы она не ухудшалась, иначе совсем беда и мы зря всё делаем.

Вторая, третья и четвёртая метрики контролируют качество работы сервиса, нам важно, чтобы они сильно не проседали, так как это может повлечь репутационные издержки и отток пользователей в будущем.

Вспомогательные метрики:

| Метрика | Аудитория | Знаменатель | Единица анализа |
| --- | --- | --- | --- |
|$\frac{\text{clicks}}{\text{impressions}}$ |  Пользователи с хотя бы 1 показом | Показы | Пользователи |
|$\frac{\text{add_to_cart}}{\text{clicks}}$ | Пользователи с хотя бы 1 кликом | Клики | Пользователи |
|$\frac{\text{purchases}}{\text{add_to_cart}}$ | Пользователи с хотя бы 1 добавлением в корзину | Добавления в корзину | Пользователи |
|$\frac{\text{orders}}{\text{users}}$ | Все пользователи | Пользователи | Пользователи |
|$\frac{\text{GMV}}{\text{orders}}$ | Пользователи с хотя бы 1 заказом | Заказы | Пользователи |


В качестве вспомогательных рассмотрим конверсии на всех этапах воронки рекомендаций. Нам хочется отслеживать, что именно изменения в рекомендательной системе, которые мы совершили, будут влиять на целевую метрику. Также будем посматривать на среднее число заказов и средний объем одного заказа

In [ ]:
# Также зафиксируем уровень ошибки первого рода
alpha = 0.05

## Оценка эффекта

Для метрик, где в показателе users будем использовать обычный t-test Уэлча

А для метрик отношений будем использовать дельта-метод

In [ ]:
def make_ttest(control, test):
    tstat, pval = ttest_ind(control, test, equal_var=False)
    c_var = np.var(control, ddof=1)
    t_var = np.var(test, ddof=1)
    n1, n2 = len(control), len(test)
    df_welch = (c_var / n1 + t_var / n2) ** 2 / ((c_var / n1) ** 2 / (n1 - 1) + (t_var / n2) ** 2 / (n2 - 1))
    se = np.sqrt(c_var / n1 + t_var / n2)
    diff = test.mean() - control.mean()
    ratio_diff = diff / control.mean()
    t_quantile = t(df=df_welch).ppf(1 - alpha / 2)
    ci = (diff - t_quantile * se, diff + t_quantile * se)
    return control.mean(), test.mean(), diff, ratio_diff, ci, pval

In [ ]:
def make_delta_method_propotions(control_numerators, control_denominators, test_numerators, test_denominators):

    def delta_method_var(numerator, denominator):
        n = len(numerator)
        num, den = np.array(numerator), np.array(denominator)
        X_mean, Y_mean = num.mean(), den.mean()
        X_sigma2 = np.var(num, ddof=1)
        Y_sigma2 = np.var(den, ddof=1)
        XY_cov = np.cov(num, den, ddof=1)[0, 1]
        variance = (1 / n) * (1 / Y_mean ** 2) * (X_sigma2 - 2 * XY_cov * X_mean / Y_mean + Y_sigma2 * X_mean ** 2 / Y_mean ** 2)
        return variance

    R_c = control_numerators.mean() / control_denominators.mean()
    R_t = test_numerators.mean() / test_denominators.mean()

    var_control = delta_method_var(control_numerators, control_denominators)
    var_test = delta_method_var(test_numerators, test_denominators)

    diff = (R_t - R_c)
    ratio_diff = diff / R_c

    se = np.sqrt(var_control + var_test)

    z_stat = (R_t - R_c) / se
    p_value = norm.cdf(-abs(z_stat)) * 2

    z_quantile = norm.ppf(1 - alpha / 2)
    ci = (diff - z_quantile * se, diff + z_quantile * se)

    return R_c, R_t, diff, ratio_diff, ci, p_value


In [ ]:
# Подготовка данных

data_control = data[data['experiment_group'] == 'control']
data_control_with_orders = data[(data['experiment_group'] == 'control') & (data['orders'] > 0)]

data_treatment = data[data['experiment_group'] == 'treatment']
data_treatment_with_orders = data[(data['experiment_group'] == 'treatment') & (data['orders'] > 0)]

dc_impressions = data_control[data_control['recommendation_impressions'] > 0]
dc_clicks = data_control[data_control['recommendation_clicks'] > 0]
dc_adds = data_control[data_control['recommended_add_to_cart'] > 0]

dt_impressions = data_treatment[data_treatment['recommendation_impressions'] > 0]
dt_clicks = data_treatment[data_treatment['recommendation_clicks'] > 0]
dt_adds = data_treatment[data_treatment['recommended_add_to_cart'] > 0]


In [ ]:
def make_line(control_value, treatment_value, diff, ratio_diff, ci, pval, line_name):
    return pd.DataFrame({
        'control value': f'{control_value:.4f}',
        'absolute diff': f'{'+' if diff > 0 else ''}{diff:.4f}',
        'ratio diff, %': f'{'+' if diff > 0 else ''}{100 * ratio_diff:.3f}',
        'confidence interval': f'({ci[0]:.3f}, {ci[1]:.3f})',
        'p-value': f'{pval:.4f}'
    }, index=[line_name])

### Целевая метрика

In [ ]:
control_value, treatment_value, diff, ratio_diff, ci, pval = make_ttest(data_control['gmv'], data_treatment['gmv'])
main_metric = make_line(control_value, treatment_value, diff, ratio_diff, ci, pval, 'GMV')

main_metric

,control value,absolute diff,"ratio diff, %",confidence interval,p-value
GMV,1173.6692,+71.5258,+6.094,"(30.164, 112.888)",0.0007


### Контрольные метрики

In [ ]:
control_metrics = []

# Прибыль
control_value, treatment_value, diff, ratio_diff, ci, pval = make_ttest(
    data_control['gross_margin'] - data_control['delivery_cost'] - data_control['promo_cost'],
    data_treatment['gross_margin'] - data_treatment['delivery_cost'] - data_treatment['promo_cost'])
control_metrics.append(make_line(control_value, treatment_value, diff, ratio_diff, ci, pval, 'Прибыль'))

# Доля отмены
control_value, treatment_value, diff, ratio_diff, ci, pval = make_delta_method_propotions(
    data_control_with_orders['cancellations'], data_control_with_orders['orders'],
    data_treatment_with_orders['cancellations'], data_treatment_with_orders['orders'],
)
control_metrics.append(make_line(control_value, treatment_value, diff, ratio_diff, ci, pval, 'Доля отмены'))

# Доля задержек в доставке
control_value, treatment_value, diff, ratio_diff, ci, pval = make_delta_method_propotions(
    data_control_with_orders['late_deliveries'], data_control_with_orders['orders'],
    data_treatment_with_orders['late_deliveries'], data_treatment_with_orders['orders'],
)
control_metrics.append(make_line(control_value, treatment_value, diff, ratio_diff, ci, pval, 'Доля задержек'))

# Доля обращений в поддержку
control_value, treatment_value, diff, ratio_diff, ci, pval = make_delta_method_propotions(
    data_control_with_orders['support_contacts'], data_control_with_orders['orders'],
    data_treatment_with_orders['support_contacts'], data_treatment_with_orders['orders'],
)
control_metrics.append(make_line(control_value, treatment_value, diff, ratio_diff, ci, pval, 'Доля обращений в поддержку'))

pd.concat(control_metrics)

,control value,absolute diff,"ratio diff, %",confidence interval,p-value
Прибыль,81.5625,-26.1285,-32.035,"(-31.070, -21.187)",0.0000
Доля отмены,0.0431,+0.0093,+21.624,"(0.003, 0.015)",0.0019
Доля задержек,0.0728,+0.0187,+25.715,"(0.011, 0.026)",0.0000
Доля обращений в поддержку,0.0425,+0.0044,+10.380,"(-0.002, 0.010)",0.1436


### Вспомогательные метрики

In [ ]:
supporting_metrics = []

# Конверсия показов в клики
control_value, treatment_value, diff, ratio_diff, ci, pval = make_delta_method_propotions(
    dc_impressions['recommendation_clicks'], dc_impressions['recommendation_impressions'],
    dt_impressions['recommendation_clicks'], dt_impressions['recommendation_impressions'],
)
supporting_metrics.append(make_line(control_value, treatment_value, diff, ratio_diff, ci, pval, 'Конверсия показов в клики'))


# Конверсия кликов в добавления
control_value, treatment_value, diff, ratio_diff, ci, pval = make_delta_method_propotions(
    dc_clicks['recommended_add_to_cart'], dc_clicks['recommendation_clicks'],
    dt_clicks['recommended_add_to_cart'], dt_clicks['recommendation_clicks'],
)
supporting_metrics.append(make_line(control_value, treatment_value, diff, ratio_diff, ci, pval, 'Конверсия кликов в добавления'))


# Конверсия добавлений в заказы
control_value, treatment_value, diff, ratio_diff, ci, pval = make_delta_method_propotions(
    dc_adds['recommended_purchases'], dc_adds['recommended_add_to_cart'],
    dt_adds['recommended_purchases'], dt_adds['recommended_add_to_cart'],
)
supporting_metrics.append(make_line(control_value, treatment_value, diff, ratio_diff, ci, pval, 'Конверсия добавлений в заказы'))

# Число заказов
control_value, treatment_value, diff, ratio_diff, ci, pval = make_ttest(
    data_control['orders'],
    data_treatment['orders'])
supporting_metrics.append(make_line(control_value, treatment_value, diff, ratio_diff, ci, pval, 'Число заказов'))


# Средний чек
control_value, treatment_value, diff, ratio_diff, ci, pval = make_delta_method_propotions(
    data_control_with_orders['gmv'], data_control_with_orders['orders'],
    data_treatment_with_orders['gmv'], data_treatment_with_orders['orders'],
)
supporting_metrics.append(make_line(control_value, treatment_value, diff, ratio_diff, ci, pval, 'Средний чек'))

pd.concat(supporting_metrics)

,control value,absolute diff,"ratio diff, %",confidence interval,p-value
Конверсия показов в клики,0.0816,+0.0190,+23.340,"(0.017, 0.022)",0.0000
Конверсия кликов в добавления,0.3653,+0.0300,+8.216,"(0.016, 0.044)",0.0000
Конверсия добавлений в заказы,0.5061,+0.0368,+7.266,"(0.013, 0.061)",0.0029
Число заказов,0.6570,+0.0200,+3.044,"(-0.001, 0.041)",0.0640
Средний чек,1786.3340,+52.8686,+2.960,"(30.008, 75.729)",0.0000


## Экономический результат

По контрольной метрике прибыли видим, что как раз на treatment происходит недопустимое (и очень статистически значимое) уменьшение прибыли. Можем проверить, какая именно компонента (`gross_margin`, `delivery_cost`, `promo_cost`) влияет на это

In [ ]:
additional_metrics = []

control_value, treatment_value, diff, ratio_diff, ci, pval = make_ttest(
    data_control['gross_margin'],
    data_treatment['gross_margin'])
additional_metrics.append(make_line(control_value, treatment_value, diff, ratio_diff, ci, pval, 'Маржа'))


control_value, treatment_value, diff, ratio_diff, ci, pval = make_ttest(
    data_control['delivery_cost'],
    data_treatment['delivery_cost'])
additional_metrics.append(make_line(control_value, treatment_value, diff, ratio_diff, ci, pval, 'Расходы на доставку'))

control_value, treatment_value, diff, ratio_diff, ci, pval = make_ttest(
    data_control['promo_cost'],
    data_treatment['promo_cost'])
additional_metrics.append(make_line(control_value, treatment_value, diff, ratio_diff, ci, pval, 'Расходы на маркетинг'))

pd.concat(additional_metrics)

,control value,absolute diff,"ratio diff, %",confidence interval,p-value
Маржа,322.3060,+4.8323,+1.499,"(-6.326, 15.990)",0.3960
Расходы на доставку,183.8401,+17.7875,+9.676,"(11.654, 23.921)",0.0000
Расходы на маркетинг,56.9034,+13.1732,+23.150,"(11.135, 15.211)",0.0000


Видим, что маржа не выросла статистически значимо, но при этом расходы выросли колоссально и по эффекту, и по статистической значимости. В этом и кроется причина падения прибыли

## Фильтр по показу

Повторим эксперименты на данных с `was_exposed == 1` и сравним с выводами по всей выборке

In [ ]:
# Подготовка данных c was_exposed == 1

data_control = data[(data['experiment_group'] == 'control') & (data['was_exposed'] == 1)]
data_control_with_orders = data[(data['experiment_group'] == 'control') & (data['orders'] > 0) & (data['was_exposed'] == 1)]

data_treatment = data[(data['experiment_group'] == 'treatment') & (data['was_exposed'] == 1)]
data_treatment_with_orders = data[(data['experiment_group'] == 'treatment') & (data['orders'] > 0) & (data['was_exposed'] == 1)]

dc_impressions = data_control[data_control['recommendation_impressions'] > 0]
dc_clicks = data_control[data_control['recommendation_clicks'] > 0]
dc_adds = data_control[data_control['recommended_add_to_cart'] > 0]

dt_impressions = data_treatment[data_treatment['recommendation_impressions'] > 0]
dt_clicks = data_treatment[data_treatment['recommendation_clicks'] > 0]
dt_adds = data_treatment[data_treatment['recommended_add_to_cart'] > 0]


In [ ]:
control_value, treatment_value, diff, ratio_diff, ci, pval = make_ttest(data_control['gmv'], data_treatment['gmv'])
main_metric = make_line(control_value, treatment_value, diff, ratio_diff, ci, pval, 'GMV')

main_metric

,control value,absolute diff,"ratio diff, %",confidence interval,p-value
GMV,1283.5482,+71.7412,+5.589,"(17.226, 126.256)",0.0099


In [ ]:
control_metrics = []

# Прибыль
control_value, treatment_value, diff, ratio_diff, ci, pval = make_ttest(
    data_control['gross_margin'] - data_control['delivery_cost'] - data_control['promo_cost'],
    data_treatment['gross_margin'] - data_treatment['delivery_cost'] - data_treatment['promo_cost'])
control_metrics.append(make_line(control_value, treatment_value, diff, ratio_diff, ci, pval, 'Прибыль'))

# Доля отмены
control_value, treatment_value, diff, ratio_diff, ci, pval = make_delta_method_propotions(
    data_control_with_orders['cancellations'], data_control_with_orders['orders'],
    data_treatment_with_orders['cancellations'], data_treatment_with_orders['orders'],
)
control_metrics.append(make_line(control_value, treatment_value, diff, ratio_diff, ci, pval, 'Доля отмены'))

# Доля задержек в доставке
control_value, treatment_value, diff, ratio_diff, ci, pval = make_delta_method_propotions(
    data_control_with_orders['late_deliveries'], data_control_with_orders['orders'],
    data_treatment_with_orders['late_deliveries'], data_treatment_with_orders['orders'],
)
control_metrics.append(make_line(control_value, treatment_value, diff, ratio_diff, ci, pval, 'Доля задержек'))

# Доля обращений в поддержку
control_value, treatment_value, diff, ratio_diff, ci, pval = make_delta_method_propotions(
    data_control_with_orders['support_contacts'], data_control_with_orders['orders'],
    data_treatment_with_orders['support_contacts'], data_treatment_with_orders['orders'],
)
control_metrics.append(make_line(control_value, treatment_value, diff, ratio_diff, ci, pval, 'Доля обращений в поддержку'))

pd.concat(control_metrics)

,control value,absolute diff,"ratio diff, %",confidence interval,p-value
Прибыль,89.6987,-27.9874,-31.202,"(-34.558, -21.417)",0.0000
Доля отмены,0.0435,+0.0079,+18.083,"(0.001, 0.015)",0.0277
Доля задержек,0.0701,+0.0229,+32.656,"(0.014, 0.032)",0.0000
Доля обращений в поддержку,0.0425,+0.0026,+6.081,"(-0.004, 0.010)",0.4675


In [ ]:
supporting_metrics = []

# Конверсия показов в клики
control_value, treatment_value, diff, ratio_diff, ci, pval = make_delta_method_propotions(
    dc_impressions['recommendation_clicks'], dc_impressions['recommendation_impressions'],
    dt_impressions['recommendation_clicks'], dt_impressions['recommendation_impressions'],
)
supporting_metrics.append(make_line(control_value, treatment_value, diff, ratio_diff, ci, pval, 'Конверсия показов в клики'))


# Конверсия кликов в добавления
control_value, treatment_value, diff, ratio_diff, ci, pval = make_delta_method_propotions(
    dc_clicks['recommended_add_to_cart'], dc_clicks['recommendation_clicks'],
    dt_clicks['recommended_add_to_cart'], dt_clicks['recommendation_clicks'],
)
supporting_metrics.append(make_line(control_value, treatment_value, diff, ratio_diff, ci, pval, 'Конверсия кликов в добавления'))


# Конверсия добавлений в заказы
control_value, treatment_value, diff, ratio_diff, ci, pval = make_delta_method_propotions(
    dc_adds['recommended_purchases'], dc_adds['recommended_add_to_cart'],
    dt_adds['recommended_purchases'], dt_adds['recommended_add_to_cart'],
)
supporting_metrics.append(make_line(control_value, treatment_value, diff, ratio_diff, ci, pval, 'Конверсия добавлений в заказы'))

# Число заказов
control_value, treatment_value, diff, ratio_diff, ci, pval = make_ttest(
    data_control['orders'],
    data_treatment['orders'])
supporting_metrics.append(make_line(control_value, treatment_value, diff, ratio_diff, ci, pval, 'Число заказов'))


# Средний чек
control_value, treatment_value, diff, ratio_diff, ci, pval = make_delta_method_propotions(
    data_control_with_orders['gmv'], data_control_with_orders['orders'],
    data_treatment_with_orders['gmv'], data_treatment_with_orders['orders'],
)
supporting_metrics.append(make_line(control_value, treatment_value, diff, ratio_diff, ci, pval, 'Средний чек'))

pd.concat(supporting_metrics)

,control value,absolute diff,"ratio diff, %",confidence interval,p-value
Конверсия показов в клики,0.0816,+0.0190,+23.340,"(0.017, 0.022)",0.0000
Конверсия кликов в добавления,0.3653,+0.0300,+8.216,"(0.016, 0.044)",0.0000
Конверсия добавлений в заказы,0.5061,+0.0368,+7.266,"(0.013, 0.061)",0.0029
Число заказов,0.7197,+0.0148,+2.061,"(-0.013, 0.043)",0.2978
Средний чек,1783.3594,+61.6443,+3.457,"(33.846, 89.443)",0.0000


Результаты тестов несильно меняются по сравнению с анализом на всех данных. Но вообще рассматривать только строчки с `was_exposed == 1` неправильно, так как значение этого поля может зависеть от применения/отсутствия эксперимента

 ## Эффект на разных сегментах

Рассмотрим основные метрики (целевую и прибыль) на разных сегментах.
`platform == "unknown"` рассматривать не будем, так как она слишком малочислена

In [ ]:
index = pd.MultiIndex.from_product(
    [['GMV', 'Прибыль'], ['android', 'ios'], ['moscow', 'other', 'spb']],
    names=['metric', 'platform', 'region']
)

segments_df = pd.DataFrame(index=index, columns=['control value', 'absolute diff', 'ratio diff, %', 'confidence interval', 'p-value'])
for indx in index:
    data_control = data[(data['experiment_group'] == 'control') & (data['platform'] == indx[1]) & (data['region'] == indx[2])]
    data_treatment = data[(data['experiment_group'] == 'treatment') & (data['platform'] == indx[1]) & (data['region'] == indx[2])]

    if indx[0] == 'GMV':
        control_value, treatment_value, diff, ratio_diff, ci, pval = make_ttest(data_control['gmv'], data_treatment['gmv'])
    else:
        control_value, treatment_value, diff, ratio_diff, ci, pval = make_ttest(
            data_control['gross_margin'] - data_control['delivery_cost'] - data_control['promo_cost'],
            data_treatment['gross_margin'] - data_treatment['delivery_cost'] - data_treatment['promo_cost']
        )

    segments_df.loc[indx, 'control value'] = f'{control_value:.4f}'
    segments_df.loc[indx, 'absolute diff'] = f'{'+' if diff > 0 else ''}{diff:.4f}'
    segments_df.loc[indx, 'ratio diff, %'] = f'{'+' if diff > 0 else ''}{100 * ratio_diff:.3f}'
    segments_df.loc[indx, 'confidence interval'] = f'({ci[0]:.3f}, {ci[1]:.3f})'
    segments_df.loc[indx, 'p-value'] = f'{pval:.4f}'

segments_df

control value absolute diff ratio diff, %  \
metric  platform region                                             
GMV     android  moscow     1292.6555      +30.0573        +2.325   
                 other      1091.7866      +63.7455        +5.839   
                 spb        1179.6870     +103.9575        +8.812   
        ios      moscow     1256.1677      +97.2117        +7.739   
                 other      1054.0034      +51.9792        +4.932   
                 spb        1123.1220     +114.9007       +10.230   
Прибыль android  moscow      125.1388      -33.1598       -26.498   
                 other        37.2381      -28.2567       -75.881   
                 spb          85.8111      -23.0691       -26.884   
        ios      moscow      119.5952      -25.6903       -21.481   
                 other        33.0951      -19.3982       -58.613   
                 spb          77.1508      -20.3050       -26.319   

                        confidence interval p-value  
metric  platform region                              
GMV     android  moscow  (-64.387, 124.502)  0.5327  
                 other   (-23.908, 151.399)  0.1540  
                 spb     (-10.470, 218.385)  0.0750  
        ios      moscow   (-5.394, 199.817)  0.0633  
                 other   (-43.048, 147.007)  0.2836  
                 spb      (-6.177, 235.978)  0.0629  
Прибыль android  moscow  (-45.537, -20.782)  0.0000  
                 other   (-36.700, -19.813)  0.0000  
                 spb      (-36.661, -9.478)  0.0009  
        ios      moscow  (-39.110, -12.270)  0.0002  
                 other   (-28.521, -10.275)  0.0000  
                 spb      (-34.461, -6.149)  0.0049

Можно заметить, что в городах кроме Москвы и СПб дела обстоят значительно хуже, чем в столицах (и прибыль уменьшается сильнее, и целевая метрика не даёт статистически значимого прироста), но тем не менее во всех сегментах прибыль даёт огромный отрицательный эффект и маленький p-value, так что решение по запуску эксперимента будет одинаковым для всех сегментов

## Решение

Несмотря на статистически значимый рост целевой метрики и конверсии на всех этапах воронки рекомендаций, мы наблюдаем сильное и также статистически значимое падение контрольной (заградительной) метрики прибыли.

Из чего вывод: **не запускать** новую ранжирующую формулу в продукт.

Основное ограничение анализа :
- невозможность интерпретировать причины роста издержек и как следствие падения прибыли, а также разницы в метриках между городами

- короткое окно анализа. Может быть, пользователи по новой рекомендательной системе будут чаще возвращаться к покупкам

Гипотезы, которые можно было бы проверить при наличии соответствующих возможностей:
- Новая ранжирующая формула слишком хорошо продвигает товары с большими скидками (и промо-затратами соответственно). Если в этом и не заключалась новая стратегия, то можно попробовать добавить штраф при ранжировании товарам с большим `promo_cost`

- Если ресурсы позволяют, попробовать продлить эксперимент и посмотреть, не улучшились ли показатели новой формулы в долгосрочной перспективе
